# CLIP-SENet VehicleID Training

VehicleID M4 training kernel triggered by the VeRi-776 CLIP-SENet v6 result (91.54 mAP with reranking). This trains from scratch on VehicleID train identities and evaluates the Small split with the standard 10-trial single-shot protocol.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Force PyTorch 2.4.1+cu124 for P100 (sm_60) compatibility; newer Kaggle defaults can drop sm_60 support.
!pip install -q --upgrade torch==2.4.1+cu124 torchvision==0.19.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q --upgrade open_clip_torch==2.30.0 timm==1.0.11
!pip install -q pretrainedmodels==0.7.4
import sys, os, time, json, hashlib, random
sys.path.insert(0, '/kaggle/working')
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith(('torch', 'torchvision')):
        del sys.modules[mod]
print("python:", sys.version)
import torch; print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "cuda_arch:", torch.cuda.get_arch_list() if torch.cuda.is_available() else None)

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)

In [ ]:
import math
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Sampler
import torchvision.transforms as T

SEED = 3407
VEHICLEID_DATASET_SLUG = 'maphat/vehicleid'
VEHICLEID_INPUT_NAME = VEHICLEID_DATASET_SLUG.split('/', 1)[1]
WORKING = '/kaggle/working'
EXTRACT_ROOT = Path(WORKING) / 'vehicleid_extracted'
CKPT_DIR = f'{WORKING}/checkpoints'
FINAL_WEIGHTS_PATH = f'{WORKING}/vehicle_clip_senet_vehicleid.pth'
FINAL_METADATA_PATH = f'{WORKING}/vehicle_clip_senet_vehicleid_metadata.json'
RESUME_INPUT_DIR = '/kaggle/input'
os.makedirs(CKPT_DIR, exist_ok=True)

CFG = {
    'seed': SEED,
    'num_classes': None,
    'image_size': (320, 320),
    'P': 8,
    'P_effective': 16,
    'K': 8,
    'accum_steps': 2,
    'batch_p': 8,
    'batch_k': 8,
    'batch_size': 64,
    'epochs': 24,
    'warmup_epochs': 5,
    'lr': 5e-4,
    'weight_decay': 5e-4,
    'label_smoothing': 0.1,
    'supcon_temperature': 0.07,
    'eval_every': 2,
    'log_every': 50,
    'max_session_hours': 11.0,
    'num_workers': 4,
    'eval_trials': 10,
    'eval_seed': 3407,
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_vehicleid_layout(base: Path) -> tuple[Path, Path, Path] | None:
    roots = [base]
    if base.exists():
        roots.extend(path for path in base.rglob('*') if path.is_dir())
    for root in roots:
        split_dir = root / 'train_test_split'
        if not split_dir.is_dir():
            continue
        if not (split_dir / 'train_list.txt').is_file() or not (split_dir / 'test_list_800.txt').is_file():
            continue
        image_candidates = [root / 'image', root / 'images', root / 'Image', root / 'Images']
        image_candidates.extend(path for path in root.iterdir() if path.is_dir() and path.name.lower() in {'image', 'images'})
        for image_dir in image_candidates:
            if image_dir.is_dir():
                return root, split_dir, image_dir
    return None


def discover_vehicleid_root() -> tuple[Path, Path, Path]:
    input_base = Path('/kaggle/input')
    print(f'discover: searching {input_base}', flush=True)
    if input_base.exists():
        for cand in input_base.rglob('image'):
            layout = find_vehicleid_layout(cand.parent)
            if layout is not None:
                return layout
        for cand in input_base.rglob('VehicleID_V1.0/image'):
            layout = find_vehicleid_layout(cand.parent)
            if layout is not None:
                return layout
        for base in sorted(path for path in input_base.glob('*') if path.is_dir()):
            layout = find_vehicleid_layout(base)
            if layout is not None:
                return layout
        for cand in input_base.rglob('train_test_split'):
            if cand.is_dir():
                root = cand.parent
                if (root / 'image').exists():
                    return root, cand, root / 'image'
    zip_candidates = sorted(input_base.rglob('VehicleID_V1.0.zip')) + sorted(input_base.rglob('*.zip'))
    if not zip_candidates:
        # Diagnostic: list everything under /kaggle/input
        print('=== DIAGNOSTIC: /kaggle/input tree ===', flush=True)
        if input_base.exists():
            for p in sorted(input_base.rglob('*'))[:200]:
                try:
                    size = p.stat().st_size if p.is_file() else 0
                    print(f'  {p} ({size} bytes)' if p.is_file() else f'  {p}/', flush=True)
                except Exception as e:
                    print(f'  {p} <error: {e}>', flush=True)
        else:
            print('  /kaggle/input does not exist', flush=True)
        print('=== END DIAGNOSTIC ===', flush=True)
        raise FileNotFoundError('Could not find VehicleID files or zip under /kaggle/input/*')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    marker = EXTRACT_ROOT / '.extracted'
    if not marker.exists():
        zip_path = zip_candidates[0]
        print(f'Extracting {zip_path} -> {EXTRACT_ROOT}')
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(EXTRACT_ROOT)
        marker.write_text(str(zip_path), encoding='utf-8')
    layout = find_vehicleid_layout(EXTRACT_ROOT)
    if layout is None:
        raise FileNotFoundError(f'Could not find image/ + train_test_split/ in extracted {EXTRACT_ROOT}')
    return layout


set_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT, SPLIT_ROOT, IMAGE_ROOT = discover_vehicleid_root()
SESSION_START_TIME = time.time()
print('DEVICE:', DEVICE)
print('DATA_ROOT:', DATA_ROOT)
print('SPLIT_ROOT:', SPLIT_ROOT)
print('IMAGE_ROOT:', IMAGE_ROOT)
print('CKPT_DIR:', CKPT_DIR)

In [ ]:
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

def build_image_index(image_root: Path) -> dict[str, Path]:
    index = {}
    for path in image_root.rglob('*'):
        if not path.is_file() or path.suffix.lower() not in IMG_EXTENSIONS:
            continue
        rel = path.relative_to(image_root).as_posix()
        index[rel] = path
        index[path.name] = path
        index[path.stem] = path
    return index


IMAGE_INDEX = build_image_index(IMAGE_ROOT)
print(f'Indexed VehicleID images: {len({str(path) for path in IMAGE_INDEX.values()})}')


def infer_pid(token: str, parts: list[str]) -> int:
    if len(parts) >= 2:
        return int(parts[1])
    path = Path(token.replace('\\', '/'))
    if path.parent.name and path.parent.name != '.':
        return int(path.parent.name)
    return int(path.stem.split('_')[0])


def resolve_image_path(token: str) -> Path:
    normalized = token.replace('\\', '/').lstrip('./')
    candidates = [normalized]
    if Path(normalized).suffix == '':
        candidates.extend([normalized + ext for ext in IMG_EXTENSIONS])
    for candidate in candidates:
        direct_paths = [IMAGE_ROOT / candidate, DATA_ROOT / candidate]
        for direct_path in direct_paths:
            if direct_path.is_file():
                return direct_path
        if candidate in IMAGE_INDEX:
            return IMAGE_INDEX[candidate]
        name = Path(candidate).name
        if name in IMAGE_INDEX:
            return IMAGE_INDEX[name]
        stem = Path(candidate).stem
        if stem in IMAGE_INDEX:
            return IMAGE_INDEX[stem]
    raise FileNotFoundError(f'Could not resolve VehicleID image token: {token}')


def load_vehicleid_list(list_name: str, relabel: bool = False) -> tuple[list[tuple[str, int, int]], int]:
    list_path = SPLIT_ROOT / list_name
    if not list_path.is_file():
        raise FileNotFoundError(f'Missing VehicleID split list: {list_path}')
    records = []
    missing = 0
    with list_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            token = parts[0]
            pid = infer_pid(token, parts)
            try:
                image_path = resolve_image_path(token)
            except FileNotFoundError:
                missing += 1
                continue
            records.append((str(image_path), pid, 0))
    if missing:
        print(f'WARNING: skipped {missing} unresolved images from {list_name}')
    unique_pids = sorted({pid for _, pid, _ in records})
    if relabel:
        pid_to_label = {pid: idx for idx, pid in enumerate(unique_pids)}
        records = [(path, pid_to_label[pid], camid) for path, pid, camid in records]
    return records, len(unique_pids)


class VehicleIDDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        image_path, pid, camid = self.records[index]
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(pid), int(camid)


train_records, num_train_ids = load_vehicleid_list('train_list.txt', relabel=True)
test_records, num_test_ids = load_vehicleid_list('test_list_800.txt', relabel=False)
CFG['num_classes'] = num_train_ids
print(f'[VehicleID] train IDs: {num_train_ids} | train images: {len(train_records)} | small-test IDs: {num_test_ids} | small-test images: {len(test_records)}')
if num_train_ids < 12000:
    print(f'WARNING: expected about 13164 VehicleID train IDs, found {num_train_ids}')
if num_test_ids < 700:
    print(f'WARNING: expected about 800 VehicleID Small test IDs, found {num_test_ids}')

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
height, width = CFG['image_size']

train_transform = T.Compose([
    T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((height, width)),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
    T.RandomErasing(p=0.5, value='random'),
])
eval_transform = T.Compose([
    T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print(json.dumps({
    'train_images': len(train_records),
    'test_images_small': len(test_records),
    'num_train_ids': num_train_ids,
    'num_test_ids_small': num_test_ids,
}, indent=2))

In [ ]:
class RandomIdentitySampler(Sampler):
    def __init__(self, records, batch_p: int, batch_k: int):
        self.records = records
        self.batch_p = batch_p
        self.batch_k = batch_k
        self.batch_size = batch_p * batch_k
        self.pid_to_indices = defaultdict(list)
        for index, (_, pid, _) in enumerate(records):
            self.pid_to_indices[int(pid)].append(index)
        self.pids = list(self.pid_to_indices.keys())
        self.length = 0
        for pid in self.pids:
            count = len(self.pid_to_indices[pid])
            if count < self.batch_k:
                count = self.batch_k
            self.length += count - (count % self.batch_k)
        self.length -= self.length % self.batch_size

    def __iter__(self):
        batch_indices = []
        batch_dict = {}
        for pid in self.pids:
            indices = list(self.pid_to_indices[pid])
            if len(indices) < self.batch_k:
                indices = list(np.random.choice(indices, size=self.batch_k, replace=True))
            random.shuffle(indices)
            pid_batches = []
            for offset in range(0, len(indices), self.batch_k):
                chunk = indices[offset: offset + self.batch_k]
                if len(chunk) < self.batch_k:
                    chunk.extend(random.choices(self.pid_to_indices[pid], k=self.batch_k - len(chunk)))
                pid_batches.append(chunk)
            batch_dict[pid] = pid_batches
        available_pids = [pid for pid, chunks in batch_dict.items() if len(chunks) > 0]
        while len(available_pids) >= self.batch_p:
            selected_pids = random.sample(available_pids, self.batch_p)
            for pid in selected_pids:
                batch_indices.extend(batch_dict[pid].pop(0))
                if len(batch_dict[pid]) == 0:
                    available_pids.remove(pid)
        return iter(batch_indices[: self.length])

    def __len__(self):
        return self.length


train_loader = DataLoader(
    VehicleIDDataset(train_records, transform=train_transform),
    batch_size=CFG['batch_size'],
    sampler=RandomIdentitySampler(train_records, CFG['batch_p'], CFG['batch_k']),
    num_workers=CFG['num_workers'],
    pin_memory=True,
    drop_last=True,
    persistent_workers=CFG['num_workers'] > 0,
)
test_loader = DataLoader(
    VehicleIDDataset(test_records, transform=eval_transform),
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
    persistent_workers=CFG['num_workers'] > 0,
)
print(f"micro-batch: P={CFG['P']} K={CFG['K']} = {CFG['P']*CFG['K']}, effective batch: {CFG['P_effective']*CFG['K']} via accum_steps={CFG['accum_steps']}")

In [ ]:
class CrossEntropyLabelSmooth(torch.nn.Module):
    def __init__(self, num_classes: int, epsilon: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = torch.nn.LogSoftmax(dim=1)

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = self.logsoftmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1.0 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        return (-targets_smooth * log_probs).mean(0).sum()


class SupConLoss(torch.nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        features = torch.nn.functional.normalize(features, p=2, dim=1)
        logits = torch.matmul(features, features.t()) / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True).values.detach()
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.t()).float().to(features.device)
        logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0), device=features.device)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)
        positives = mask.sum(dim=1)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / torch.clamp(positives, min=1.0)
        valid = positives > 0
        if not torch.any(valid):
            return features.new_tensor(0.0)
        return -mean_log_prob_pos[valid].mean()


def build_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])


def build_scheduler(optimizer):
    def lr_lambda(epoch_idx: int) -> float:
        if epoch_idx < CFG['warmup_epochs']:
            return float(epoch_idx + 1) / float(max(1, CFG['warmup_epochs']))
        progress = float(epoch_idx - CFG['warmup_epochs'] + 1) / float(max(1, CFG['epochs'] - CFG['warmup_epochs']))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


model = build_clip_senet(num_classes=CFG['num_classes']).to(DEVICE)
torch.backends.cudnn.benchmark = True
param_count = sum(parameter.numel() for parameter in model.parameters())
print('loaded_tinyclip_model:', model.loaded_tinyclip_model)
print('loaded_appearance_model:', model.loaded_resnext_model)
print('param_count:', param_count)

ce_criterion = CrossEntropyLabelSmooth(num_classes=CFG['num_classes'], epsilon=CFG['label_smoothing']).to(DEVICE)
supcon_criterion = SupConLoss(temperature=CFG['supcon_temperature']).to(DEVICE)
optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
scaler = GradScaler(enabled=DEVICE == 'cuda')
state = {
    'start_epoch': 0,
    'completed_epochs': 0,
    'best_R1': 0.0,
    'best_R5': 0.0,
    'best_epoch': 0,
    'history': [],
    'stopped_early': False,
}

In [ ]:
def capture_rng_state() -> dict:
    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch': torch.get_rng_state(),
        'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state_dict: dict | None) -> None:
    if not state_dict:
        return
    random.setstate(state_dict['python'])
    np.random.set_state(state_dict['numpy'])
    torch.set_rng_state(state_dict['torch'])
    if torch.cuda.is_available() and state_dict.get('cuda') is not None:
        torch.cuda.set_rng_state_all(state_dict['cuda'])


def find_resume_checkpoint() -> Path | None:
    input_root = Path(RESUME_INPUT_DIR)
    if not input_root.exists():
        return None
    resume_candidates = sorted(set(input_root.glob('*/checkpoints/last.pth')))
    if not resume_candidates:
        return None
    best_path = None
    best_epoch = -1
    for candidate in resume_candidates:
        try:
            checkpoint = torch.load(candidate, map_location='cpu', weights_only=False)
            epoch_idx = int(checkpoint.get('epoch', -1))
        except Exception as exc:
            print(f'SKIPPING CHECKPOINT {candidate}: {exc}')
            continue
        if epoch_idx > best_epoch:
            best_epoch = epoch_idx
            best_path = candidate
    return best_path


def save_checkpoint(model, optimizer, scheduler, scaler, epoch_idx, state, metrics):
    payload = {
        'epoch': epoch_idx,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'best_R1': state['best_R1'],
        'best_R5': state['best_R5'],
        'best_epoch': state['best_epoch'],
        'history': state['history'],
        'last_metrics': metrics,
        'rng_state': capture_rng_state(),
    }
    last_path = Path(CKPT_DIR) / 'last.pth'
    torch.save(payload, last_path)
    if metrics is not None and metrics.get('R1', -1.0) >= state['best_R1'] and metrics.get('is_best', False):
        torch.save(payload, Path(CKPT_DIR) / 'best.pth')
    print(f'SAVED CHECKPOINT: {last_path}')


def train_one_epoch(model, loader, optimizer, scaler, ce_criterion, supcon_criterion, epoch_idx):
    model.train()
    running = {'loss': 0.0, 'ce': 0.0, 'supcon': 0.0, 'steps': 0}
    accum_steps = CFG['accum_steps']
    num_steps = len(loader)
    log_every = CFG.get('log_every', 50)
    optimizer.zero_grad(set_to_none=True)
    for step_idx, (images, pids, _) in enumerate(loader, start=1):
        images = images.to(DEVICE, non_blocking=True)
        pids = pids.to(DEVICE, non_blocking=True)
        with autocast(enabled=DEVICE == 'cuda', dtype=torch.float16):
            features, logits = model(images)
            ce_loss = ce_criterion(logits, pids)
            supcon_loss = supcon_criterion(features, pids)
            total_loss = ce_loss + supcon_loss
            loss = total_loss / accum_steps
        scaler.scale(loss).backward()
        if step_idx % accum_steps == 0 or step_idx == num_steps:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running['loss'] += float(total_loss.item())
        running['ce'] += float(ce_loss.item())
        running['supcon'] += float(supcon_loss.item())
        running['steps'] += 1
        if step_idx == 1 or step_idx % log_every == 0 or step_idx == num_steps:
            print(f'EPOCH {epoch_idx + 1:02d} STEP {step_idx:04d} loss={total_loss.item():.4f} ce={ce_loss.item():.4f} supcon={supcon_loss.item():.4f} lr={optimizer.param_groups[0]["lr"]:.7f}', flush=True)
    steps = max(1, running['steps'])
    return {key: value / steps for key, value in running.items() if key != 'steps'}


def train_model(model, optimizer, scheduler, scaler, ce_criterion, supcon_criterion, state):
    session_limit_reached = False
    for epoch_idx in range(state['start_epoch'], CFG['epochs']):
        epoch_start = time.time()
        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, ce_criterion, supcon_criterion, epoch_idx)
        scheduler.step()
        metrics = {
            'epoch': epoch_idx + 1,
            'train_loss': train_metrics['loss'],
            'train_ce': train_metrics['ce'],
            'train_supcon': train_metrics['supcon'],
            'R1': state['best_R1'],
            'R5': state['best_R5'],
            'is_best': False,
        }
        if (epoch_idx + 1) % CFG['eval_every'] == 0 or epoch_idx == CFG['epochs'] - 1:
            eval_metrics = evaluate_vehicleid(model, test_loader, test_records, trials=CFG['eval_trials'])
            metrics.update(eval_metrics)
            if eval_metrics['R1'] > state['best_R1']:
                state['best_R1'] = float(eval_metrics['R1'])
                state['best_R5'] = float(eval_metrics['R5'])
                state['best_epoch'] = epoch_idx + 1
                metrics['is_best'] = True
            print(f'EVAL epoch={epoch_idx + 1:02d} R1={eval_metrics["R1"]:.4f} R5={eval_metrics["R5"]:.4f} valid_ids={eval_metrics["valid_ids"]}')
        elapsed_epoch = time.time() - epoch_start
        metrics['epoch_seconds'] = elapsed_epoch
        state['history'].append(metrics)
        state['start_epoch'] = epoch_idx + 1
        state['completed_epochs'] = epoch_idx + 1
        save_checkpoint(model, optimizer, scheduler, scaler, epoch_idx, state, metrics)
        session_hours = (time.time() - SESSION_START_TIME) / 3600.0
        if session_hours >= CFG['max_session_hours'] and epoch_idx + 1 < CFG['epochs']:
            session_limit_reached = True
            print(f'SESSION LIMIT REACHED at epoch {epoch_idx + 1:02d}; exiting cleanly for resume.')
            break
    state['stopped_early'] = session_limit_reached
    return state


resume_path = find_resume_checkpoint()
if resume_path is not None:
    checkpoint = torch.load(resume_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    if checkpoint.get('scaler_state') is not None:
        scaler.load_state_dict(checkpoint['scaler_state'])
    restore_rng_state(checkpoint.get('rng_state'))
    state['start_epoch'] = int(checkpoint['epoch']) + 1
    state['completed_epochs'] = int(checkpoint['epoch']) + 1
    state['best_R1'] = float(checkpoint.get('best_R1', 0.0))
    state['best_R5'] = float(checkpoint.get('best_R5', 0.0))
    state['best_epoch'] = int(checkpoint.get('best_epoch', 0))
    state['history'] = list(checkpoint.get('history', []))
    print(f'RESUMING from epoch {state["start_epoch"] + 1} using {resume_path}')
else:
    print('STARTING FRESH')

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    features, pids = [], []
    for images, batch_pids, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        batch_features = model(images)
        features.append(batch_features.cpu())
        pids.extend(batch_pids.tolist())
    return torch.cat(features, dim=0), np.asarray(pids)


def evaluate_vehicleid(model, loader, records, trials: int = 10):
    features, pids = extract_features(model, loader)
    pid_to_indices = defaultdict(list)
    for index, pid in enumerate(pids.tolist()):
        pid_to_indices[int(pid)].append(index)
    valid_pids = [pid for pid, indices in pid_to_indices.items() if len(indices) >= 2]
    if not valid_pids:
        raise RuntimeError('No VehicleID identities with at least two images for single-shot evaluation')

    rng = np.random.default_rng(CFG['eval_seed'])
    r1_scores, r5_scores = [], []
    for trial_idx in range(trials):
        gallery_indices, query_indices = [], []
        for pid in sorted(valid_pids):
            indices = np.asarray(pid_to_indices[pid], dtype=np.int64)
            gallery_index = int(rng.choice(indices))
            gallery_indices.append(gallery_index)
            query_indices.extend([int(index) for index in indices if int(index) != gallery_index])
        gallery_indices = np.asarray(gallery_indices, dtype=np.int64)
        query_indices = np.asarray(query_indices, dtype=np.int64)
        gallery_features = features[gallery_indices]
        query_features = features[query_indices]
        gallery_pids = pids[gallery_indices]
        query_pids = pids[query_indices]
        sim = torch.matmul(query_features, gallery_features.t()).numpy()
        order = np.argsort(-sim, axis=1)
        matches = gallery_pids[order] == query_pids[:, None]
        r1_scores.append(float(matches[:, :1].any(axis=1).mean()))
        r5_scores.append(float(matches[:, :5].any(axis=1).mean()))
        print(f'VEHICLEID SMALL TRIAL {trial_idx + 1:02d}/{trials}: R1={r1_scores[-1]:.4f} R5={r5_scores[-1]:.4f} queries={len(query_indices)} gallery={len(gallery_indices)}')
    return {
        'R1': float(np.mean(r1_scores)),
        'R5': float(np.mean(r5_scores)),
        'R1_std': float(np.std(r1_scores)),
        'R5_std': float(np.std(r5_scores)),
        'valid_ids': int(len(valid_pids)),
        'trials': int(trials),
    }

In [ ]:
def export_artifacts(model, state):
    best_ckpt_path = Path(CKPT_DIR) / 'best.pth'
    export_source = 'current_model'
    export_state_dict = {key: value.detach().cpu() for key, value in model.state_dict().items()}
    if best_ckpt_path.exists():
        best_checkpoint = torch.load(best_ckpt_path, map_location='cpu', weights_only=False)
        export_state_dict = best_checkpoint['model_state']
        export_source = 'best.pth'
    torch.save(export_state_dict, FINAL_WEIGHTS_PATH)
    metadata = {
        'dataset_slug': VEHICLEID_DATASET_SLUG,
        'dataset_split': 'VehicleID Small / test_list_800.txt',
        'epochs_trained': int(state['completed_epochs']),
        'best_epoch': int(state['best_epoch']),
        'best_R1': float(state['best_R1']),
        'best_R5': float(state['best_R5']),
        'stopped_early': bool(state['stopped_early']),
        'export_source': export_source,
        'loaded_tinyclip_model': getattr(model, 'loaded_tinyclip_model', 'unknown'),
        'loaded_appearance_model': getattr(model, 'loaded_resnext_model', 'unknown'),
        'param_count': int(sum(parameter.numel() for parameter in model.parameters())),
        'history': state['history'],
        'hyperparameters': {
            'image_size': list(CFG['image_size']),
            'batch_p': CFG['batch_p'],
            'batch_k': CFG['batch_k'],
            'batch_size': CFG['batch_size'],
            'accum_steps': CFG['accum_steps'],
            'effective_batch': CFG['P_effective'] * CFG['K'],
            'epochs': CFG['epochs'],
            'warmup_epochs': CFG['warmup_epochs'],
            'lr': CFG['lr'],
            'weight_decay': CFG['weight_decay'],
            'label_smoothing': CFG['label_smoothing'],
            'supcon_temperature': CFG['supcon_temperature'],
            'eval_every': CFG['eval_every'],
            'eval_trials': CFG['eval_trials'],
            'seed': CFG['seed'],
        },
    }
    with open(FINAL_METADATA_PATH, 'w', encoding='utf-8') as handle:
        json.dump(metadata, handle, indent=2, ensure_ascii=True)
    print(f'EXPORTED WEIGHTS: {FINAL_WEIGHTS_PATH}')
    print(f'EXPORTED METADATA: {FINAL_METADATA_PATH}')
    print(json.dumps(metadata, indent=2))
    return metadata


if state['start_epoch'] >= CFG['epochs']:
    print('TRAINING ALREADY COMPLETE; exporting artifacts only.')
else:
    state = train_model(model, optimizer, scheduler, scaler, ce_criterion, supcon_criterion, state)

export_metadata = export_artifacts(model, state)
summary = {
    'completed_epochs': state['completed_epochs'],
    'best_epoch': state['best_epoch'],
    'best_R1': state['best_R1'],
    'best_R5': state['best_R5'],
    'stopped_early': state['stopped_early'],
    'resume_ready_checkpoint': str(Path(CKPT_DIR) / 'last.pth'),
}
print('FINAL SUMMARY')
print(json.dumps(summary, indent=2))